# Lecture 4

## 4.1 Motivation

Solving many-body quantum problems, e.g. electronic structure of atoms

$$H = \sum_{i} \frac{p_i^2}{2m} + \underbrace{V(r_i)}_{\text{Coulomb pot. of nucleus}} + \underbrace{\sum_{i<j} \frac{e^2}{|r_i - r_j|}}_{e^--e^- \text{ interaction}}$$


Typically, we are interested in the ground state, or low excited states, which determine electronic/chemical properties.

$$\Phi_g(\vec{r}_1, \vec{r}_2, \dots \vec{r}_N) \quad \text{High-dimensional object}$$

Grid-based methods quickly reach their limitations due to exponential growth of memory:

$Q$ grid points per dimension $\rightarrow Q^{3N}$ grid points in total $\hat{=}$ exponential growth of Hilbert space in $N$.
> "curse of dimensionality"

$\rightarrow$ To still solve many-body problems, we need to make suitable approximations.

This lecture provides a brief overview of common approaches to numerical simulation of quantum many-body problems (necessarily incomplete).

## 4.2 Basis set expansion (and truncation)

Representing $\hat{H}$ in a suitable basis set much can often be gained compared to spatial/momentum grid methods.

The (time dep. or stationary) Schrödinger eq. can be represented in any basis $\{e_k\}_{k=1}^d$ of Hilbert space (or a subspace of the full Hilbert space, as we consider truncated bases, finite basis).

$$i \partial_t |\psi\rangle = \hat{H} |\psi\rangle \quad \longrightarrow \quad \hat{H} |\psi\rangle = E |\psi\rangle$$

$$\rightarrow i \partial_t \underbrace{\langle e_k |\psi\rangle}_{c_k} = \langle e_k | \hat{H} |\psi\rangle$$

$$c_k = \sum_{k'} \underbrace{\langle e_k | \hat{H} | e_{k'} \rangle}_{H_{k k'}} \underbrace{\langle e_{k'} |\psi\rangle}_{c_{k'}}$$

$$i \vec{\dot{c}} = H \vec{c} \quad \text{Matrix form problem in } \mathbb{C}^d \quad \quad H \vec{c} = E \vec{c}$$

**Goal:** Choose basis cleverly such that small $d$ is suff. to capture the ground state / low lying states correctly.

### Typical situation:
$$H = H_0 + H_1$$
$$\uparrow$$
$$\text{eigenstates are known } \{e_k\}$$

If $H_1 = \lambda V$ where $\lambda$ is a small parameter, we could apply standard pert. theory to the states and energies.

**Or numerically:** represent $H$ in eigenbasis $\{e_k\}$ of $H_0$ and diagonalize $\rightarrow$ can expect fast conv. of ground state energy with size of basis.

$$\langle e_k | H | e_{k'} \rangle = E_k^{(0)} \delta_{k k'} + \langle e_k | H_1 | e_{k'} \rangle$$

Challenge: Calculating matrix elements $\langle e_k | H_1 | e_{k'} \rangle$

<u>example</u>: anharmonic oscillator:

$$H = \underbrace{\left( -\frac{1}{2}\partial_x^2 + \frac{1}{2}x^2 \right)}_{H_0} + \lambda x^4 \quad \quad \begin{pmatrix} \text{oscillator units} \\ m=1, \hbar=1, \dots \end{pmatrix}$$

$\{e_k\} = \{|n\rangle\}$ harmonic osc. eigenstates

$\langle n | x^4 | n'\rangle = \text{ ?} \quad \rightarrow \text{use ladder operators!}$

$$x = (a + a^\dagger)/\sqrt{2} \quad \quad \left(a = (x + ip)/\sqrt{2} \text{ , } a^\dagger = (x - ip)/\sqrt{2}\right)$$
$$a^\dagger |n\rangle = \sqrt{n+1} |n+1\rangle$$
$$a |n\rangle = \sqrt{n} |n-1\rangle$$

$$\Rightarrow x^4 |n\rangle = \frac{1}{4} (a + a^\dagger)(a + a^\dagger)(a + a^\dagger)(a + a^\dagger) |n\rangle$$
$$= \frac{1}{4} (a + a^\dagger)(a + a^\dagger)(a + a^\dagger) \left( \sqrt{n}|n-1\rangle + \sqrt{n+1}|n+1\rangle \right)$$
$$= \dots \quad \text{complete as exercise!}$$

$$\Rightarrow \langle n' | x^4 | n \rangle = \dots$$

$$\Rightarrow H_{n n'} = \underbrace{\left(n + \frac{1}{2}\right)}_{E_n^{(0)}} \delta_{n n'} + \begin{bmatrix}
0 & \cdot & 0 & \cdot & 0 \\
\cdot & 0 & \cdot & 0 & \cdot \\
0 & \cdot & 0 & \cdot & 0 \\
\cdot & 0 & \cdot & 0 & \cdot \\
0 & \cdot & 0 & \cdot & 0 
\end{bmatrix} \begin{matrix} \text{only off-} \\ \text{diag.} \\ 0, \pm 2, \pm 4 \\ \text{non-zero} \end{matrix}$$

$$\rightarrow \text{easy to diagonalize}$$

Another general guideline, when dealing with many-body problems is to exploit symmetries: choose basis that reflects symmetries of $H$, so dim. reduces, e.g.

### H-atom 

$3\text{D} \rightarrow 1\text{D}$
(spher. coords, sph. harm.)

---

### $\text{H}_2^+$ molecule

9\text{D} \xrightarrow{\text{CM coords}} 6\text{D} \xrightarrow{\text{BO approx}} 3\text{D} \xrightarrow{\text{sym}} 2\text{D} \xrightarrow{\substack{\text{coord} \\ \text{trafo}}} 1\text{D}$

---
### He-atom
$6\text{D} \rightarrow 4\text{D}$
rot. sym.

## 4.3 Reduce dimensionality through separation of scales

### Example: Born-Oppenheimer approximation

Molecular Hamiltonian:

$$H = \underbrace{H_{\text{kin}}^{\text{nucl}}}_{\sum_i \frac{P_i^2}{2m_i}} + H_{\text{kin}}^{\text{el}} + \underbrace{\sum_{i<j} V(\vec{R}_i, \vec{R}_j) + \sum_{i, \text{nucl}} V(\vec{R}_{i, \text{nucl}}, \vec{r}_{i, \text{el}}) + \sum_{i<j} V(\vec{r}_i, \vec{r}_j)}_{H_{\text{nucl}}^{\text{pot}} \quad \text{(Coulomb int. betw. all constituents (nuclei and electrons))}}$$

$$\uparrow$$
$$\text{moments of nuclei}$$

$$V \propto \frac{1}{|\vec{r}_i - \vec{r}_j|}$$

$$m_{\text{nucl}} \gg m_e \quad \Rightarrow \quad \text{Separation of scales, electronic motion follows nuclear motion adiabatically}$$

$$\Psi(\vec{R}, \vec{r}) = \psi(\vec{R}) \phi(\vec{r}; \vec{R})$$
$$\begin{matrix} \uparrow & \uparrow & \quad \quad \quad \uparrow \\ \text{nucl. coords} & \text{electron coords} & \quad \quad \quad \text{parametric dependence} \end{matrix}$$

1. Treat nuclear coords as parameters in $H$ and solve the electronic problem. $H_{\text{nucl}}^{\text{pot}}$ becomes a static external potential seen by electrons. 
   Determine eigenvalues $E_j(\vec{R})$ as function of $\vec{R}$.
   $\rightarrow$ energy surfaces, BO potential surfaces.

2. Born-Oppenheimer surfaces $E_j(\vec{R})$ serve as potential for the nuclear problem $\rightarrow$ solve nuclear problem $\rightarrow$ bound states, rotational/vibrational excitations.
   $$H_{\text{nucl}} = \sum_i \frac{P_i^2}{2m_i} + E_j(\vec{R})$$


## 4.4 Variational methods

Problem of finding ground state of $H$ can be formulated as optimization problem:

$$E_g = \min_{|\psi\rangle} \langle \psi | H | \psi \rangle \quad \text{subject to } \langle \psi | \psi \rangle = 1$$

Make an ansatz for $|\psi\rangle$ :  $|\psi\rangle = |\psi_{\text{var}}(\theta_1, \dots \theta_M)\rangle$
$$\begin{matrix} \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \uparrow \\ \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \text{variational parameters} \end{matrix}$$

Minimize energy functional w.r.t. var. params:

$$E_g \le \min_{\theta_1, \dots \theta_M} \langle \psi_{\text{var}} | H | \psi_{\text{var}} \rangle \quad \text{subj. to } \langle \psi_{\text{var}} | \psi_{\text{var}} \rangle = 1$$

$|\psi_{\text{var}}\rangle$ parameterizes a certain variational manifold in Hilbert space (not all of it) $\rightarrow$ variational energy minimum gives <u>upper bound</u> on ground state energy.

<u>example</u>: $\psi(x; c) = \frac{1}{(\pi c)^{1/4}} e^{-x^2 / 2c}$ as variational ansatz for quartic oscillator
* $\rightarrow$ gives exact sol. in the limit of HO ($\lambda = 0$)
* $\rightarrow$ for small $\lambda$, captures well the deformation of $|\psi_g\rangle$

> **Note:** Gaussians (multi-variate) are indeed a common choice e.g. in quantum chemistry. There even exists a software library called "Gaussian".

The stationarity condition (min. cond.) can be given as the variation:

$$\delta G[\theta_1, \dots \theta_M] = \delta \left( \langle \psi_{\text{var}} | H | \psi_{\text{var}} \rangle - E \langle \psi_{\text{var}} | \psi_{\text{var}} \rangle \right) = 0$$

where $E$ is a Lagrange multiplier ensuring normalization (also often called $\mu$).

$$\Rightarrow \text{solve } \frac{\delta G}{\delta \theta_i} = 0 \quad \text{(syst. of eqs.)}$$

> More in part on spin systems (lecture 9 ff)

## 4.5 Mean field methods

Consider e.g. the multi-electron problem from 4.1

In the non-interaction case we have for the ground state:

$$\Psi(\vec{r}_1, \dots \vec{r}_N) = \psi_1(\vec{r}_1)\psi_2(\vec{r}_2)\dots\psi_N(\vec{r}_N)$$

$$\text{product state (up to symmetrization)}$$

Use this as variational ansatz for the interacting problem. Should be suitable if interactions are weak (electron-correlations can be neglected).

Breaks down in strongly correlated/interacting regimes.

* **Electronic problems** $\rightarrow$ Fermions $\rightarrow$ anti-symmetrization $\rightarrow$ Slater determinants $\rightarrow$ Hartree Fock method $\begin{pmatrix} \text{see} \\ \text{solid state} \\ \text{lecture} \end{pmatrix}$
  More advanced methods: Density functional theory, dyn. mean field theo.

* **Bosons** (e.g. bosonic atoms in a trap) $\rightarrow$ symmetric wave fct. $\rightarrow$ Bose Einstein condensate (BEC) (all atoms in same state/orbital) $\rightarrow$ Gross-Pitaevskii equation (GPE)


### Derivation of GPE:

$$H = \sum_i \underbrace{\left( \frac{P_i^2}{2m} + V(r_i) \right)}_{h_i^{(1)}} + \underbrace{\sum_{i<j} g \delta(r_i - r_j)}_{\text{"contact interactions"}}$$

$$g = \frac{4\pi \hbar^2 a_s}{m}$$
$$\begin{matrix} \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \uparrow \\ \quad \quad \quad \quad \quad \quad \quad \quad \quad \text{s-wave scattering length} \end{matrix}$$

Describes bosonic atoms at low temperatures and low density
$$\lambda_{\text{dB}} = \frac{h}{mv} \gg a_s \quad \quad \text{(mean free path } \gg \text{ scatt. cross sect.)}$$

**Ansatz:** $\Psi(r_1, \dots r_N) = \psi(r_1)\dots\psi(r_N) \quad \text{symmetric } \checkmark$

**Goal:** Find single particle wfct $\psi$ that minimizes the energy functional. ($\psi$ normalized)

$$
\begin{aligned}
E[\Psi] &= \int \psi^*(r_1) \dots \psi^*(r_N) \left[ \sum_i h_i^{(1)} + \sum_{i<j} h_{ij}^{(2)} \right] \psi(r_1) \dots \psi(r_N) \, dr_1 \dots dr_N \\
&= \sum_i \int dr_1 \dots dr_N \, \psi^*(r_1)\psi(r_1) \dots \psi^*(r_i) h_i^{(1)} \psi(r_i) \dots \psi^*(r_N)\psi(r_N) \\
&\quad + \sum_{i<j} \int dr_i \, dr_j \, |\psi_1|^2 \dots \psi^*(r_i)\psi^*(r_j) h_{ij}^{(2)} \psi(r_i)\psi(r_j) \dots |\psi_N|^2 \\
&= N \int dr \, \psi^*(r) \left( -\frac{\hbar^2}{2m} \partial_r^2 + V(r) \right) \psi(r) \\
&\quad + \frac{N(N-1)}{2} \underbrace{\int dr \, dr' \, \psi^*(r)\psi^*(r') \, g\delta(r-r') \, \psi(r)\psi(r')}_{\int dr \, |\psi(r)|^4}
\end{aligned}
$$

Define the "condensate wave function" $\phi(r) = \sqrt{N} \psi(r)$ such that $|\phi(r)|^2 = n(r)$ is the particle density of the BEC which is normalized to the total particle number: $\int n(r) \, dr = N$.

With this, the variational functional including the Lagrange mult. for the normalization const. is:

$$G[\phi] = \int dr \, \phi^* \left( -\frac{\hbar^2}{2m} \partial_r^2 + V(r) \right) \phi + \frac{g}{2} |\phi|^4 - \mu |\phi|^2$$

$$\frac{\delta G}{\delta \phi^2} = 0 \quad \rightarrow \quad \left[ -\frac{\hbar^2}{2m} \partial_r^2 + V(r) + g|\phi|^2 \right] \phi = \mu \phi \quad \quad (*)$$
$$\begin{matrix} \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \uparrow \\ \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \text{mean field = pot. caused by} \\ \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \text{other atoms forming a backgr.} \end{matrix}$$

<u>non-linear equation</u> for $\phi$

Solve by fixed point iteration: initial guess for $\phi$, e.g., non-int ($g=0$) solution $\rightarrow$ calculate mean field $g|\phi|^2$, solve eq. for $\phi$ with fixed mean field potential $\rightarrow$ iterate until convergence $\Rightarrow$ <u>self-consistent</u> mean field method.

$(*)$ = Gross-Pitaevskii eq. or non-linear Schrödinger eq.

> **Note:** One can also derive a time-dependent version of this by minimizing the action functional
> 
> $$S = \int dt \, dr \, \Psi^* (i \hbar \partial_t - H) \Psi$$
> 
> with the product ansatz

$$\Rightarrow i \hbar \partial_t \phi(r, t) = \left( -\frac{\hbar^2}{2m} \vec{\nabla}^2 + V(r) + g |\phi|^2 \right) \phi(r, t)$$

The initial value problem can be solved numerically using e.g. the split-step method, see lecture 3.